# Awaaz — Deepfake Defense (XLSR-SLS) · Colab runner

**Rules (full manual: `docs/DEEPFAKE_TRACKER.md`)**
1. Run **MASTER SEQUENCE** first in *every* new session: new runtime, after a timeout, after switching Google account.
2. Code lives in the repo; never edit code in Colab. Change it in the repo, push, re-run MASTER SEQUENCE.
3. Durable data lives in Drive (`MyDrive/AwaazDeepfake`). `/content` is scratch and disappears on disconnect.
4. Every step cell is **safe to re-run**: finished steps leave a marker in `AwaazDeepfake/state/` and skip themselves.
5. Runtime > Change runtime type > **T4 GPU** (CPU is enough for P1/P2 data steps, which saves GPU quota).

## ▶ MASTER SEQUENCE — run every session

In [ ]:
# ===================== MASTER SEQUENCE =====================
# Safe to re-run at any time. Syncs code from GitHub, mounts Drive, restores local cache, prints status.
BRANCH   = "feature/deepfake-defense"
REPO_URL = "https://github.com/Haseeb-Sultan-NU/AwaazOnBoard-Voice-Based-Authentication-System.git"
REPO     = "/content/repo"

import os, subprocess, sys
from google.colab import drive, userdata
drive.mount("/content/drive")

def _secret(name):
    try:
        return userdata.get(name)          # Colab sidebar > Secrets (key icon), per Google account
    except Exception:
        return None

gh = _secret("GITHUB_TOKEN")               # only needed while the repo is private
hf = _secret("HF_TOKEN")                   # optional (avoids HF anonymous rate limits)
if hf:
    os.environ["HF_TOKEN"] = hf
url = REPO_URL.replace("https://", f"https://x-access-token:{gh}@") if gh else REPO_URL

def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(gh, "***") if gh else (r.stdout + r.stderr)
    if r.returncode:
        raise RuntimeError(out)
    return out

if not os.path.isdir(f"{REPO}/.git"):
    run(["git", "clone", "--depth", "1", "-b", BRANCH, url, REPO])
else:  # repo is the source of truth: local edits in /content/repo are discarded on purpose
    run(["git", "-C", REPO, "remote", "set-url", "origin", url])
    run(["git", "-C", REPO, "fetch", "--depth", "1", "origin", BRANCH])
    run(["git", "-C", REPO, "reset", "--hard", f"origin/{BRANCH}"])
print("Code @", run(["git", "-C", REPO, "log", "-1", "--format=%h  %s  (%cr)"]).strip())

run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO}/deepfake/requirements-colab.txt"])
if REPO not in sys.path:
    sys.path.insert(0, REPO)
!python {REPO}/deepfake/scripts/session_setup.py

## ⚙ ONE-TIME · INIT DRIVE (home account only, first time ever)
Creates `MyDrive/AwaazDeepfake`. Afterwards share that folder (**Editor**) with every other Google account you will use.
Then, in each of those accounts: Drive web > *Shared with me* > right-click `AwaazDeepfake` > *Organize > Add shortcut* > *My Drive*.

In [ ]:
!python /content/repo/deepfake/scripts/session_setup.py --init

## P1.1 · Download LUMS Urdu deepfake dataset → Drive (one time, ~2 GB)
HF → local SSD → one `.tar` → verified copy at `AwaazDeepfake/raw/lums_urdu.tar`. CPU runtime is fine.

In [ ]:
!python /content/repo/deepfake/scripts/download_lums.py

## P1.2 · Inventory + manifest
Writes `AwaazDeepfake/manifests/lums_all.csv` and `lums_inventory.json`.
**Copy the block between the `=====` lines and paste it to Claude.** It is used to freeze the speaker split (P2).

In [ ]:
!python /content/repo/deepfake/scripts/inventory.py

## P2.1 · Apply the frozen speaker split (done once)
Reads `deepfake/splits/lums_split.json`, writes `manifests/lums_split.csv`, prints the shortcut check.

In [ ]:
!python /content/repo/deepfake/scripts/apply_split.py

## P2.2 · Preprocess + build frozen train / dev / eval sets (E1·E2·E3) → Drive
CPU runtime is fine (~20–40 min for 6.8k clips; resumable: re-run after a disconnect). Builds **eval first**, then dev, then train. Copy the `=====` block back to Claude. **Never delete `processed/eval_*.tar`.**

In [ ]:
!python /content/repo/deepfake/scripts/build_eval.py

## P3 · Baseline (zero-shot) evaluation — **T4 GPU runtime**
Run MASTER SEQUENCE first (on the GPU runtime). P3.1 and P3.2 also work on CPU, but P3.3 needs the GPU, so do all three in one T4 session.

### P3.1 · Download pretrained XLSR-SLS weights → Drive (one time, 1.36 GB, sha256-pinned)

In [ ]:
!python /content/repo/deepfake/scripts/download_pretrained.py

### P3.2 · Parity check: our model port vs the published ONNX model (one time, ~5 min)
Must print `PARITY: PASS`. If it says FAIL, stop and paste the block to Claude.

In [ ]:
!python /content/repo/deepfake/scripts/parity_check.py

### P3.3 · Score DEV + frozen EVAL with the pretrained model (resumable, ~20–40 min on T4)
If Colab disconnects: MASTER SEQUENCE, then re-run this cell; it continues where it stopped. Paste the `=====` block to Claude.

In [ ]:
!python /content/repo/deepfake/scripts/evaluate.py --run baseline

## P5 · Fine-tuning — **T4 GPU runtime**
**Every session:** MASTER SEQUENCE → the Drive-login cell below → the train cell. The login lets the trainer permanently delete *its own* replaced checkpoints from Drive Trash (otherwise each 3.9 GB save stays in Trash and fills your 15 GB). It touches nothing else.

In [ ]:
# Drive login for checkpoint Trash cleanup (popup: pick the SAME account whose Drive is mounted)
from google.colab import auth
auth.authenticate_user()

### P5.0 · Smoke test (~10 min, run once): speed, GPU memory, checkpoint save + reload, Trash purge
Must end with `SMOKE: PASS` and `Drive trash purge: ok`. Paste the block to Claude.

In [ ]:
!python /content/repo/deepfake/scripts/train.py --smoke

### P5.1 · Training run `ft1` (LUMS train speakers only; resumable)
Re-run this same cell after any disconnect / account switch: it resumes from the last checkpoint (≤ 30 min lost). Stopping the cell saves a checkpoint first. Ends with a `=====` block to paste.
If you see `CUDA out of memory`: add `--grad-ckpt` **on a new run name** (e.g. `--run ft1b`).

In [ ]:
!python /content/repo/deepfake/scripts/train.py --run ft1

### P6 · Evaluate the fine-tuned model on the SAME frozen sets (after training finishes)

In [ ]:
!python /content/repo/deepfake/scripts/evaluate.py --run ft1 --ckpt /content/drive/MyDrive/AwaazDeepfake/checkpoints/ft1/ft1_best.pt

## Next: P4 ElevenLabs set (EVAL-EL) + optional run `ft2` (LUMS + EL-adapt speakers)